<a href="https://colab.research.google.com/github/BaseballCV/BaseballCV/blob/main/notebooks/base_detector_retraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Base Detector Retraining — dataset v2.1 (#183)

Retrains the base detector on **dataset v2.1**: the same 660 broadcast frames as v1, but with 120 recovered base annotations (machine-mined, then visually verified across two mining rounds — see `datasets/yolo/base_detect_dataset_v2/README.md`) and a proper 90/10 train/val split, which v1 never had.

**What this notebook does** (Runtime > Run all; GPU runtime recommended):

1. Downloads the v1 images via `LoadTools` and the curated v2.1 labels from this repo
2. Assembles the YOLO-format dataset (classes: `base`, `distractor`)
3. Fine-tunes YOLOv11s from COCO-pretrained weights
4. Reports validation mAP per class and compares the old vs new model on the same split, plus a false-positive regression
5. Exports the trained weights for review

This addresses the retraining half of [#183](https://github.com/BaseballCV/BaseballCV/issues/183). The distractor class is intentionally small (8 verified boxes across 6 images: shoes/feet, sliding legs, pitching rubber, foul-line chunk) — a seed for future distractor annotation, not a complete fix; the interim `BaseDetector` inference mitigations remain in place.

In [ ]:
!pip install -q "baseballcv>=0.1.31"

import shutil
from pathlib import Path

import matplotlib.pyplot as plt
from ultralytics import YOLO

from baseballcv.functions import LoadTools

WORK = Path("/content/work"); WORK.mkdir(exist_ok=True)
load_tools = LoadTools()
print("Setup OK")

## 1. Data: v1 images + curated v2 labels

In [ ]:
# v1 images (vault-hosted) and the current production weights, for comparison later
images_dir = Path(str(load_tools.load_dataset('base_detect_dataset'))) / "images" / "train"
old_weights = load_tools.load_model('base_detector')
print(f"{len(list(images_dir.glob('*')))} images | old weights at {old_weights}")

# curated v2 labels from this repo (sparse clone — labels only, fast)
from IPython import get_ipython
REPO = Path("/content/repo")
if not REPO.exists():
    get_ipython().system('git clone -q --depth 1 --filter=blob:none --sparse https://github.com/BaseballCV/BaseballCV.git "{REPO}"')
get_ipython().system('git -C "{REPO}" sparse-checkout set datasets/yolo/base_detect_dataset_v2')
V2 = REPO / "datasets/yolo/base_detect_dataset_v2"
print("v2 labels:", len(list((V2 / 'labels/train').glob('*.txt'))), "train +",
      len(list((V2 / 'labels/val').glob('*.txt'))), "val")

In [ ]:
# Assemble the YOLO dataset: images from the vault, labels from the curated set
DS = WORK / "base_detect_v2"
for split in ("train", "val"):
    (DS / "images" / split).mkdir(parents=True, exist_ok=True)
    (DS / "labels" / split).mkdir(parents=True, exist_ok=True)
    for lf in (V2 / "labels" / split).glob("*.txt"):
        stem = lf.stem
        src = next((p for p in images_dir.glob(stem + ".*")), None)
        assert src is not None, f"image not found for label {stem}"
        shutil.copy(src, DS / "images" / split / src.name)
        shutil.copy(lf, DS / "labels" / split / lf.name)
(DS / "data.yaml").write_text(
    f"path: {DS.resolve().as_posix()}\ntrain: images/train\nval: images/val\n"
    "names:\n  0: base\n  1: distractor\n")
print("dataset assembled at", DS)

## 2. Sanity-check a few labels

In [ ]:
import cv2
import numpy as np

val_imgs = sorted((DS / "images" / "val").glob("*"))
tiles = []
for p in val_imgs[:8]:
    img = cv2.imread(str(p)); ih, iw = img.shape[:2]
    lf = DS / "labels" / "val" / (p.stem + ".txt")
    for line in lf.read_text().splitlines():
        c, xc, yc, w, h = line.split()[:5]
        xc, yc, w, h = float(xc)*iw, float(yc)*ih, float(w)*iw, float(h)*ih
        color = (0, 255, 0) if int(c) == 0 else (0, 165, 255)
        cv2.rectangle(img, (int(xc-w/2), int(yc-h/2)), (int(xc+w/2), int(yc+h/2)), color, 2)
    tiles.append(cv2.resize(img, (426, 240)))
rows = [np.hstack(tiles[i:i+4]) for i in range(0, len(tiles), 4)]
plt.figure(figsize=(17, 5*len(rows)))
plt.imshow(np.vstack(rows)[:, :, ::-1]); plt.axis("off"); plt.show()

## 3. Train

Fine-tunes `yolo11s` (same architecture as the production `base_detector`) from COCO-pretrained weights so the comparison against the current model is apples-to-apples. Roughly 20 minutes on a T4 GPU for 100 epochs at this dataset size.

In [ ]:
model = YOLO("yolo11s.pt")  # auto-downloads COCO-pretrained weights
model.train(
    data=str(DS / "data.yaml"),
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    name="base_detector_v21",
)
BEST = Path(model.trainer.best).resolve()
print("best weights:", BEST)

## 4. Evaluate: new model vs production model on the same val split

The production model has no `distractor` class and its own training data overlaps this split, so treat these numbers as a same-split sanity check rather than a held-out benchmark. The honest question is: does the new model detect bases at least as well while producing fewer false base detections?

In [ ]:
new_model = YOLO(str(BEST))
new_metrics = new_model.val(data=str(DS / "data.yaml"), verbose=False)

old_model = YOLO(old_weights)
old_metrics = old_model.val(data=str(DS / "data.yaml"), classes=[0], verbose=False)

# box.ap50 / box.maps are per-class AP50 / mAP50-95 (position 0 = class 0, base)
print(f"{'':<16}{'OLD (production)':>18}{'NEW (v2.1)':>12}")
print(f"{'base mAP50':<16}{float(old_metrics.box.map50):>18.4f}{float(new_metrics.box.ap50[0]):>12.4f}")
print(f"{'base mAP50-95':<16}{float(old_metrics.box.map):>18.4f}{float(new_metrics.box.maps[0]):>12.4f}")

# False-positive regression (the #183 failure mode): base-class detections at
# conf 0.5 with no IoU >= 0.3 overlap with any ground-truth base
def _iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / ua if ua else 0.0

def fp_scan(m):
    tp = fp = 0
    for img in sorted((DS / "images" / "val").glob("*")):
        r = m.predict(str(img), conf=0.5, verbose=False)[0]
        h, w = r.orig_shape
        gt = []
        for line in (DS / "labels" / "val" / (img.stem + ".txt")).read_text().splitlines():
            p = line.split()
            if len(p) >= 5 and int(p[0]) == 0:
                xc, yc, bw, bh = (float(v) for v in p[1:5])
                gt.append((xc*w - bw*w/2, yc*h - bh*h/2, xc*w + bw*w/2, yc*h + bh*h/2))
        for b in r.boxes:
            if int(b.cls) != 0:
                continue
            xyxy = tuple(b.xyxy[0].tolist())
            if any(_iou(xyxy, g) >= 0.3 for g in gt):
                tp += 1
            else:
                fp += 1
    return tp, fp

for label, m in [("OLD (production)", old_model), ("NEW (v2.1)", new_model)]:
    tp, fp = fp_scan(m)
    print(f"{label}: {tp} true positives | {fp} false positives @ conf 0.5")

In [ ]:
# Side-by-side predictions on a few val images
fig, axes = plt.subplots(3, 2, figsize=(16, 13))
for ax_pair, p in zip(axes, [str(p) for p in val_imgs[:3]]):
    for ax, (label, m) in zip(ax_pair, [("OLD (production)", old_model), ("NEW (v2)", new_model)]):
        r = m.predict(p, conf=0.5, verbose=False)[0]
        ax.imshow(r.plot()[:, :, ::-1]); ax.set_title(label); ax.axis("off")
plt.tight_layout(); plt.show()

## 5. Export weights

In [ ]:
shutil.copy(str(BEST), str(WORK / "base_detector_v21.pt"))
print("artifact:", WORK / "base_detector_v21.pt")
from google.colab import files
files.download(str(WORK / "base_detector_v21.pt"))

## Next steps after training

1. Download `base_detector_v2.pt` and get it to the maintainers (PR referencing #183, or the contacts in the README).
2. Maintainer review flow: check the val mAP and in-notebook FP regression above, run the `BaseDetector` FP checks on mound shots and slide footage, then publish to the vault as `base_detector_v2` — or replace `base_detector` if it strictly dominates.
3. Grow the distractor class: mine mound-visible frames from broadcast clips, auto-label rubber candidates with the retrained model, verify visually (same workflow used for the v2 labels — about 5 minutes per batch).